first project

In [10]:
import os
#Team members:
#Gemma Mena, 1745878
#Alicja Dorobis, 1736872


# Exercise 1: Implement DocClass_Welcome()

In [11]:
def DocClass_Welcome(): 
    message = 'Document Classification is typically used in Document Processing for assigning categories to documents. Thus, it eases their posterior processing and analysis.\nThe program “DocumentClassification” classifies an input folder containing documents in txt format. It will also receive a file containing information about the topics to classify.\nThe program will generate the following files\nA txt file containing the category of the documents\nA txt file containing the statistics of the classification\n'
    return message

In [12]:
print(DocClass_Welcome())

Document Classification is typically used in Document Processing for assigning categories to documents. Thus, it eases their posterior processing and analysis.
The program “DocumentClassification” classifies an input folder containing documents in txt format. It will also receive a file containing information about the topics to classify.
The program will generate the following files
A txt file containing the category of the documents
A txt file containing the statistics of the classification



# Exercise 2: Implement Read_Categories_Keywords()


In [13]:
def Read_Categories_Keywords(file_name):
    with open (file_name, "r") as cat_file:
        LCategories = []
        LKeywords = []
        for line in cat_file:
            first_word = line.strip().split()[0][:-1]
            LCategories.append(first_word)
            
            keyword_set = set(line.strip().split(" ")[1:])
            LKeywords.append(keyword_set)
            
    return LCategories, LKeywords


In [14]:
#main

#print(Read_Categories_Keywords("ToyExample/keywords_toyExample.txt"))

# Exercise 3: Implement Document_Reader()

In [15]:
def Document_Reader(filename):
    with open(filename) as file:
    
        text = file.read()
        
    text = text.lower()    
    LWords = []
    current_word = []
        
    valid = 'abcdefghijklmnñopqrstuvwxyz0123456789\''
        
    for char in text:
        if char in valid:
            current_word.append(char)
        else:                        
            if current_word:
                LWords.append(''.join(current_word))
                current_word = []
    if current_word:
        LWords.append(''.join(current_word))   
    return LWords

# Exercise 4: Implement Count_Frequencies()

In [16]:
def Count_Frequencies(LWords, LCategories, LKeywords):
    
    frequencies = {}
    
    for i, category in enumerate(LCategories):
        frequencies[category] = {keyword: 0 for keyword in LKeywords[i]}
        
        for word in LWords:
            if word in frequencies[category]:
                frequencies[category][word] += 1
    return frequencies 

# Exercise 5: Implement Classify_Doc()

For example, if DFrequencies is
• {'medical': {'physician': 0, 'infection': 0, 'patient': 0, 'care': 0},
'space': {'planet': 1, 'orbit': 0, 'nasa': 2},
'sports': {'medal': 0, 'record': 0, 'score': 0, 'team': 1, 'training': 0}
}

the output will be (3 occurrences of category 'space'
, so 75% and 1 occurrence of 'sports'
, so 25%):
• DVotes = {'medical': (0,0), 'space': (3,75), 'sports': (1,25)}

In [17]:
def Classify_Doc(DFrequencies):
    #parameter initilisation
    DVotes = {}
    total_count = 0 
    max_percentage = 0
    max_category = None

    # calculating the counts
    for category in DFrequencies: #iterating over the categories in the dictionary
        category_count = 0

        for keyword in DFrequencies[category]: # iterating over the keywords in each category
            keyword_count = DFrequencies[category][keyword]
            category_count +=  keyword_count

        total_count += category_count
        DVotes[category] = (category_count,0)

    
    #calculating the percentages
    for category in DVotes:
        count = DVotes[category][0]

        if total_count <= 0:
            percentage = 0
        else:
            percentage = int(count/total_count * 100)
            DVotes[category] = (count, percentage)
        
        if percentage > max_percentage:
            max_percentage = percentage
            max_category = category

     
    #checking if any category has a percentage higher than 50
    if max_percentage > 50: 
        DocCategory = max_category
    else:
        DocCategory = "UNKNOWN" #setting the category to "UNKNOWN"
    
    return DVotes, DocCategory

DFrequencies = {'medical': {'physician': 0, 'infection': 0, 'patient': 0, 'care': 0},
'space': {'planet': 1, 'orbit': 0, 'nasa': 2},
'sports': {'medal': 0, 'record': 0, 'score': 0, 'team': 1, 'training': 0}
}
print(Classify_Doc(DFrequencies))

({'medical': (0, 0), 'space': (3, 75), 'sports': (1, 25)}, 'space')


# Exercise 6: Implement main():

In [24]:
def main():
    DocClass_Welcome()
    
    while True:
        file_name = input("Enter the name of the file with categories and keywords")
        # checking if file exists and contains any data
        if os.path.exists(file_name) and os.path.getsize(file_name) > 0:
            LCategories,LKeywords = Read_Categories_Keywords(file_name)
            break
        else:
            print("File does not exist or is empty. Try again.")

    while True:
        folder_name = input("Enter the name of the folder that contains the document files to be processed:")
        if os.path.exists(folder_name) and os.path.isdir(folder_name) > 0:

            #initialising variables for statistics
            category_counts = {}
            total_files_number = 0
            unknown_count = 0
            keyword_counts = [] #list of tuples

            with open(f"{folder_name}_DocClassification.txt", "w") as file_output:

                for file_n_name in os.listdir(folder_name):
                        if file_n_name.endswith('.txt'):
                            file_path = os.path.join(folder_name, file_n_name)
                            LWords = Document_Reader(file_path)
                            DFrequencies = Count_Frequencies(LWords,LCategories,LKeywords)
                            DVotes,DocCategory = Classify_Doc(DFrequencies)

                            total_files_number +=1 #counting the total number of files
                        
                            if DocCategory != "UNKNOWN":
                                file_output.write("{file_n_name}: {DocCategory}: {DVotes}")

                                if DocCategory not in category_counts:
                                    category_counts[DocCategory] = 0
                                category_counts[DocCategory] += 1

                            else: #output when the category is "UNKNOWN"
                                file_output.write("{file_n_name}: {DocCategory}")
                                unknown_count+= 1
                                
                        else:
                            file_output.write(f"Attention: the following non txt file has been detected:{file_n_name}")

                with open(f"{folder_name}_DocStatistics.txt", "w") as stats_file:
                    stats_file.write("*** Statistics\n")
                    stats_file.write(f"Total txt files: {total_files_number}\n")

                    for category, count in category_counts.items(): #iterating over the dictionary with category counts
                        percentage = (count / total_files_number) * 100  #calculating the percentage occurance of each category
                        stats_file.write(f"{category}: {percentage:.2f}%\n")
                    stats_file.write(f"UNKNOWN: {unknown_count/total_files_number:.2f}%\n")

                    
                    for keyword, count in keywords.items():
                        # checking if the keyword exists in the list
                        found = False
                        for i in range(len(keyword_counts)):
                            existing_keyword, existing_count = keyword_counts[i]
                            if existing_keyword == keyword:
                                # updateing the count
                                keyword_counts[i] = (existing_keyword, existing_count + count)
                                found = True
                                break
                        if not found:
                            # adding new keyword to the list
                            keyword_counts.append((keyword, count))


                    #computing the top 10 keywords
                    stats_file.write("*** Top 10 keywords\n")

                    #sorting the keywords
                    top_keywords = sorted(keyword_counts, key=lambda x: x[1], reverse=True)[:10]

                    # writing the top keywords to the file
                    for keyword, count in top_keywords:
                        stats_file.write(f"{keyword}: {count}\n")



if __name__ == "__main__":
    main()



UnicodeDecodeError: 'utf-8' codec can't decode byte 0x93 in position 1378: invalid start byte

In [ ]:
keywords_toyExample.txt